# Irish Property Price Register - SQL Analysis

## Overview
This notebook loads the cleaned Irish housing data into a SQLite database 
and runs analytical SQL queries to answer business questions about the 
Irish property market.

## Objectives
1. Create a structured database from the cleaned CSV data
2. Answer business questions using SQL queries
3. Export key results for use in Power BI dashboard

## Tech Stack
- Python (data loading and cleaning)
- SQLite (database and SQL queries)
- Pandas (displaying results)

In [64]:
import pandas as pd
import sqlite3
from pathlib import Path

# Load the cleaned data
DATA_PATH = Path("PPR-ALL.csv")
df = pd.read_csv(DATA_PATH, encoding="latin-1")

# Clean the data exactly as in the main notebook
df['Price'] = df.iloc[:, 4].str.replace('\x80', '').str.replace(',', '').str.strip()
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Date'] = pd.to_datetime(df['Date of Sale (dd/mm/yyyy)'], dayfirst=True, errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['County'] = df['County'].str.strip().str.title()
df['Not Full Market Price'] = df['Not Full Market Price'].str.strip()
df_clean = df[df['Not Full Market Price'] == 'No'].copy()
df_clean = df_clean.dropna(subset=['Price', 'Date'])
df_clean = df_clean[df_clean['Price'] <= 5000000].copy()

# Simplify property type labels
df_clean['Property Type'] = df_clean['Description of Property'].str.strip()
df_clean['Property Type'] = df_clean['Property Type'].replace({
    'Second-Hand Dwelling house /Apartment': 'Second-Hand',
    'New Dwelling house /Apartment': 'New Build',
    'Teach/Árasán Cónaithe Atháimhe': 'Second-Hand',
    'Teach/Árasán Cónaithe Nua': 'New Build',
    'Teach/?ras?n C?naithe Nua': 'New Build'
})

print(f"Records loaded: {df_clean.shape[0]:,}")

C:\Users\navya\AppData\Local\Temp\ipykernel_12076\4049622795.py:7: DtypeWarning: Columns (0: Property Size Description) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, encoding="latin-1")


Records loaded: 748,051


## Step 1 - Create SQLite Database

We load the cleaned data into a SQLite database, creating a structured 
table that we can query using standard SQL.

In [65]:
# Create a SQLite database and load the cleaned data into it
conn = sqlite3.connect('housing.db')

# Write the cleaned dataframe to a SQL table called 'transactions'
df_clean[['Date', 'County', 'Year', 'Month', 'Price', 'Property Type', 'Eircode']].to_sql(
    'transactions', 
    conn, 
    if_exists='replace', 
    index=False
)

# Verify the table was created correctly
result = pd.read_sql("SELECT COUNT(*) as total_records FROM transactions", conn)
print(f"Records in database: {result['total_records'][0]:,}")
print("Database created successfully - ready for SQL queries")

Records in database: 748,051
Database created successfully - ready for SQL queries


## Step 2 - SQL Queries

We now answer 10 business questions about the Irish property market 
using SQL queries on the database.

### Query 1 - National Average Price and Transaction Volume by Year

This query groups all transactions by year and calculates the average 
price and total number of sales, giving us a high-level view of how 
the market has changed since 2010.

In [66]:
query1 = """
SELECT 
    Year,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price
FROM transactions
GROUP BY Year
ORDER BY Year
"""
result1 = pd.read_sql(query1, conn)
print("National Price Trends by Year:")
print(result1.to_string(index=False))

National Price Trends by Year:
 Year  total_transactions  avg_price
 2010               19899   245871.0
 2011               17344   219203.0
 2012               23664   197976.0
 2013               28475   199508.0
 2014               42526   207053.0
 2015               47478   216941.0
 2016               47972   238107.0
 2017               52315   265615.0
 2018               54102   280112.0
 2019               55205   285237.0
 2020               46810   291242.0
 2021               56904   317496.0
 2022               59724   348395.0
 2023               59623   362266.0
 2024               57350   391403.0
 2025               58625   415516.0
 2026               20035   422981.0


Average property prices fell sharply from €245,871 in 2010 to a low of 
€197,976 in 2012, before recovering steadily to €422,981 by 2026 - a 
72% increase from the post-crash low. Transaction volumes tell a similar 
story, dropping to 17,344 in 2011 before recovering to a peak of 59,724 
in 2022.

### Query 2 - Average Price by County (Latest Year)

This query filters to the most recent year and calculates average price, 
minimum price, and maximum price for each county, ranked from most to 
least expensive. This gives a current snapshot of regional affordability 
across all 26 Irish counties.

In [67]:
query2 = """
SELECT 
    County,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price,
    ROUND(MIN(Price), 0) as min_price,
    ROUND(MAX(Price), 0) as max_price
FROM transactions
WHERE Year = (SELECT MAX(Year) FROM transactions)
GROUP BY County
ORDER BY avg_price DESC
"""

result2 = pd.read_sql(query2, conn)
print("County Price Analysis (Latest Year):")
print(result2.to_string(index=False))

County Price Analysis (Latest Year):
   County  total_transactions  avg_price  min_price  max_price
   Dublin                6026   591104.0    13000.0  5000000.0
  Wicklow                 699   534350.0    20000.0  4000000.0
  Kildare                1151   440878.0    15000.0  2350000.0
    Meath                 852   420534.0    30000.0  2550000.0
     Cork                2440   389516.0     8400.0  4915000.0
   Galway                 959   385928.0    10000.0  2632708.0
 Kilkenny                 349   355888.0     7620.0  2200000.0
    Louth                 633   346927.0    60000.0  1200000.0
    Laois                 425   343113.0    20000.0  4395783.0
    Kerry                 407   327947.0    30000.0  3752395.0
Waterford                 565   317401.0    30000.0  3476212.0
  Wexford                 763   317041.0    30000.0  3488326.0
    Clare                 468   312290.0    26150.0  2600000.0
 Limerick                 683   310293.0    25000.0  3835000.0
Westmeath         

Dublin is the most expensive county with an average price of €591,104, 
more than double Longford's €204,414. The top 4 most expensive counties 
(Dublin, Wicklow, Kildare, Meath) are all within the Greater Dublin Area 
commuter belt, confirming the strong influence of proximity to Dublin on 
property prices.

### Query 3 - Year on Year Price Growth

This query uses the SQL window function LAG() to compare each year's 
average price to the previous year, calculating both the absolute change 
in euros and the percentage change. This shows us which years saw the 
strongest growth and which saw price falls.

In [68]:
query3 = """
WITH yearly_prices AS (
    SELECT
        Year,
        AVG(Price) as avg_price
    FROM transactions
    GROUP BY Year
)
SELECT
    Year,
    ROUND(avg_price, 0) as avg_price,
    ROUND(avg_price - LAG(avg_price) OVER (ORDER BY Year), 0) as yoy_change,
    ROUND(
        (avg_price - LAG(avg_price) OVER (ORDER BY Year))
        / LAG(avg_price) OVER (ORDER BY Year) * 100,
        1
    ) as yoy_pct_change
FROM yearly_prices
ORDER BY Year
"""

result3 = pd.read_sql(query3, conn)
print("Year on Year Price Growth:")
print(result3.to_string(index=False))


Year on Year Price Growth:
 Year  avg_price  yoy_change  yoy_pct_change
 2010   245871.0         NaN             NaN
 2011   219203.0    -26668.0           -10.8
 2012   197976.0    -21226.0            -9.7
 2013   199508.0      1531.0             0.8
 2014   207053.0      7545.0             3.8
 2015   216941.0      9888.0             4.8
 2016   238107.0     21166.0             9.8
 2017   265615.0     27508.0            11.6
 2018   280112.0     14497.0             5.5
 2019   285237.0      5126.0             1.8
 2020   291242.0      6004.0             2.1
 2021   317496.0     26254.0             9.0
 2022   348395.0     30899.0             9.7
 2023   362266.0     13871.0             4.0
 2024   391403.0     29137.0             8.0
 2025   415516.0     24114.0             6.2
 2026   422981.0      7465.0             1.8



The steepest price falls were in 2011 (-10.8%) and 2012 (-9.7%) following 
the financial crash. The strongest recovery years were 2017 (+11.6%) and 
2022 (+9.7%). Notably, prices continued rising even through 2020 (+2.1%) 
despite COVID-19, suggesting demand remained resilient throughout the 
pandemic period.

### Query 4 - Dublin vs National Average Price by Year

This query uses conditional aggregation (CASE WHEN) to calculate Dublin's 
average price and the national average price side by side for each year, 
then subtracts one from the other to show the Dublin premium over time.

In [69]:
query4 = """
SELECT 
    Year,
    ROUND(AVG(CASE WHEN County = 'Dublin' THEN Price END), 0) as dublin_avg,
    ROUND(AVG(Price), 0) as national_avg,
    ROUND(AVG(CASE WHEN County = 'Dublin' THEN Price END) - AVG(Price), 0) as dublin_premium
FROM transactions
GROUP BY Year
ORDER BY Year
"""

result4 = pd.read_sql(query4, conn)
print("Dublin vs National Average Price by Year:")
print(result4.to_string(index=False))

Dublin vs National Average Price by Year:
 Year  dublin_avg  national_avg  dublin_premium
 2010    331272.0      245871.0         85401.0
 2011    313264.0      219203.0         94061.0
 2012    284779.0      197976.0         86802.0
 2013    304071.0      199508.0        104563.0
 2014    337255.0      207053.0        130202.0
 2015    349430.0      216941.0        132489.0
 2016    385361.0      238107.0        147255.0
 2017    416960.0      265615.0        151345.0
 2018    432546.0      280112.0        152434.0
 2019    429181.0      285237.0        143944.0
 2020    442299.0      291242.0        151058.0
 2021    485975.0      317496.0        168479.0
 2022    515295.0      348395.0        166901.0
 2023    521589.0      362266.0        159323.0
 2024    554947.0      391403.0        163544.0
 2025    578055.0      415516.0        162539.0
 2026    591104.0      422981.0        168123.0


The Dublin premium grew from €85,401 in 2010 to a peak of €168,479 in 
2021 - nearly doubling over 11 years. The premium has slightly narrowed 
since 2021 (down to €162,539 in 2025), which may reflect remote working 
enabling buyers to consider properties outside Dublin, though Dublin 
remains by far the most expensive county.

### Query 5 - New Build vs Second-Hand Price by Year

This query uses conditional aggregation to separate new build and 
second-hand prices for each year, and calculates the premium (or 
discount) that new builds commanded relative to second-hand properties. 
A negative value means second-hand was more expensive that year.

In [70]:
query5 = """
SELECT 
    Year,
    ROUND(AVG(CASE WHEN [Property Type] = 'New Build' THEN Price END), 0) as new_build_avg,
    ROUND(AVG(CASE WHEN [Property Type] = 'Second-Hand' THEN Price END), 0) as second_hand_avg,
    ROUND(AVG(CASE WHEN [Property Type] = 'New Build' THEN Price END) - 
          AVG(CASE WHEN [Property Type] = 'Second-Hand' THEN Price END), 0) as new_build_premium
FROM transactions
GROUP BY Year
ORDER BY Year
"""

result5 = pd.read_sql(query5, conn)
print("New Build vs Second-Hand Price by Year:")
print(result5.to_string(index=False))

New Build vs Second-Hand Price by Year:
 Year  new_build_avg  second_hand_avg  new_build_premium
 2010       214015.0         257070.0           -43055.0
 2011       190310.0         224798.0           -34488.0
 2012       174035.0         201474.0           -27439.0
 2013       154080.0         206400.0           -52320.0
 2014       186557.0         209989.0           -23432.0
 2015       221638.0         216243.0             5396.0
 2016       265882.0         233627.0            32255.0
 2017       293979.0         259646.0            34333.0
 2018       320359.0         270343.0            50016.0
 2019       327515.0         275423.0            52092.0
 2020       352767.0         277390.0            75377.0
 2021       368826.0         307955.0            60872.0
 2022       386063.0         340505.0            45558.0
 2023       407345.0         351901.0            55444.0
 2024       429003.0         381681.0            47322.0
 2025       429359.0         411416.0           

A significant turning point occurred in 2015, when new builds crossed 
above second-hand prices for the first time (new build premium of €5,396). 
Before 2015 second-hand properties were consistently more expensive, 
reflecting post-crash distressed new build sales. The premium peaked at 
€75,377 in 2020 and has since narrowed to €15,212 in 2026, suggesting 
second-hand prices are catching up with new builds in recent years.

### Query 6 - Top 10 Most Expensive Counties by Transaction Volume

This query identifies which counties had the highest number of transactions 
in the most recent year, showing where market activity is concentrated.

In [71]:
query6 = """
SELECT 
    County,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price,
    ROUND(MIN(Price), 0) as min_price,
    ROUND(MAX(Price), 0) as max_price
FROM transactions
GROUP BY County
ORDER BY total_transactions DESC
LIMIT 10
"""

result6 = pd.read_sql(query6, conn)
print("Top 10 Counties by Transaction Volume (All Years):")
print(result6.to_string(index=False))

Top 10 Counties by Transaction Volume (All Years):
   County  total_transactions  avg_price  min_price  max_price
   Dublin              235429   443132.0     5250.0  5000000.0
     Cork               82350   269184.0     5031.0  4990000.0
  Kildare               39786   320194.0     6348.0  4977500.0
   Galway               36458   251591.0     5500.0  4874987.0
    Meath               32067   291689.0     5500.0  5000000.0
 Limerick               27934   212007.0     6349.0  5000000.0
  Wexford               25843   210414.0     5500.0  4989971.0
  Wicklow               25382   381749.0     6248.0  5000000.0
    Louth               20884   230878.0     7000.0  4414097.0
Waterford               20202   207757.0     6000.0  3850000.0


Dublin dominates market activity with 235,429 transactions over the full 
period - nearly three times more than Cork in second place (82,350). 
Dublin and Cork together account for approximately 43% of all transactions 
in the dataset. Notably, Wicklow has a relatively low transaction volume 
(25,382) but the second highest average price (€381,749), reflecting its 
premium positioning as a coastal county close to Dublin. Limerick and 
Wexford both have high transaction volumes but average prices well below 
the national level, indicating active but more affordable markets.

### Query 7 - Price Bands Distribution

This query categorises all transactions into price bands to show how 
properties are distributed across different price ranges, giving a 
clearer picture of market accessibility for buyers at different 
budget levels.

In [72]:
query7 = """
SELECT 
    CASE 
        WHEN Price < 100000 THEN 'Under €100k'
        WHEN Price < 200000 THEN '€100k - €200k'
        WHEN Price < 300000 THEN '€200k - €300k'
        WHEN Price < 400000 THEN '€300k - €400k'
        WHEN Price < 500000 THEN '€400k - €500k'
        ELSE 'Over €500k'
    END as price_band,
    COUNT(*) as transaction_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 1) as pct_of_total
FROM transactions
GROUP BY price_band
ORDER BY MIN(Price)
"""

result7 = pd.read_sql(query7, conn)
print("Price Band Distribution:")
print(result7.to_string(index=False))

Price Band Distribution:
   price_band  transaction_count  pct_of_total
  Under €100k             101257          13.5
€100k - €200k             182595          24.4
€200k - €300k             180963          24.2
€300k - €400k             127460          17.0
€400k - €500k              67826           9.1
   Over €500k              87950          11.8


The majority of Irish property transactions fall below €300,000:

- **Under €100k:** 13.5% of transactions - likely distressed sales, 
rural properties, or older transactions from the post-crash period
- **€100k - €200k:** 24.4% - the single largest price band, reflecting 
the large number of affordable transactions outside Dublin
- **€200k - €300k:** 24.2% - broadly similar to the €100k-€200k band, 
reflecting the national median price range
- **€300k - €400k:** 17.0% - properties in this range represent the 
mid-to-upper market outside Dublin
- **€400k - €500k:** 9.1% - approaching Dublin median price territory
- **Over €500k:** 11.8% - premium properties, heavily concentrated in 
Dublin and Wicklow

Over 62% of all transactions fell below €300,000, confirming that the 
high national average is heavily influenced by Dublin's premium pricing. 
The majority of Irish buyers outside Dublin are transacting in a 
significantly more affordable market.

### Query 8 - Monthly Seasonality Analysis

This query examines whether property sales are seasonal - are there 
certain months where more transactions occur or prices are higher? 
This is useful for buyers and sellers timing their market entry.

In [73]:
query8 = """
SELECT 
    Month,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price
FROM transactions
GROUP BY Month
ORDER BY Month
"""

result8 = pd.read_sql(query8, conn)

# Add month names for readability
month_names = {1:'January', 2:'February', 3:'March', 4:'April', 
               5:'May', 6:'June', 7:'July', 8:'August', 
               9:'September', 10:'October', 11:'November', 12:'December'}
result8['Month_Name'] = result8['Month'].map(month_names)

print("Monthly Seasonality Analysis:")
print(result8[['Month_Name', 'total_transactions', 'avg_price']].to_string(index=False))

Monthly Seasonality Analysis:
Month_Name  total_transactions  avg_price
   January               44994   302977.0
  February               53587   296501.0
     March               59389   297376.0
     April               54742   295669.0
       May               59480   294444.0
      June               58647   295487.0
      July               66276   299272.0
    August               59669   306142.0
 September               63946   303891.0
   October               69571   304352.0
  November               68226   300593.0
  December               89524   294285.0


The Irish property market shows clear seasonal patterns in transaction volume:

- **December** is by far the busiest month with 89,524 transactions - 
likely driven by buyers rushing to complete purchases before year end 
for tax and mortgage reasons
- **October and November** are also high activity months (69,571 and 
68,226 respectively), forming a strong autumn selling season
- **January** is the quietest month with only 44,994 transactions - 
consistent with a post-Christmas slowdown in market activity
- **Average prices** are relatively stable across months, ranging from 
€294,285 in December to €306,142 in August - a narrow range of just 
€11,857

The high December transaction volume with relatively lower average prices 
suggests that year-end completions may include more lower-value properties 
being rushed through, while August shows the highest average price despite 
mid-range transaction volumes, potentially reflecting a premium summer 
market.

### Query 9 - County Price Growth Ranking

This query calculates which counties have seen the strongest price growth 
between 2015 and 2025, identifying the fastest growing property markets 
in Ireland over the past decade.

In [74]:
query9 = """
SELECT 
    a.County,
    ROUND(a.avg_price_2015, 0) as avg_price_2015,
    ROUND(b.avg_price_2025, 0) as avg_price_2025,
    ROUND(b.avg_price_2025 - a.avg_price_2015, 0) as absolute_growth,
    ROUND((b.avg_price_2025 - a.avg_price_2015) / a.avg_price_2015 * 100, 1) as pct_growth
FROM 
    (SELECT County, AVG(Price) as avg_price_2015 
     FROM transactions WHERE Year = 2015 
     GROUP BY County) a
JOIN 
    (SELECT County, AVG(Price) as avg_price_2025 
     FROM transactions WHERE Year = 2025 
     GROUP BY County) b
ON a.County = b.County
ORDER BY pct_growth DESC
"""

result9 = pd.read_sql(query9, conn)
print("County Price Growth 2015 to 2025:")
print(result9.to_string(index=False))

County Price Growth 2015 to 2025:
   County  avg_price_2015  avg_price_2025  absolute_growth  pct_growth
 Longford         72440.0        207969.0         135529.0       187.1
Westmeath        118793.0        325930.0         207138.0       174.4
Roscommon         82138.0        215922.0         133784.0       162.9
    Cavan         95989.0        251903.0         155914.0       162.4
    Clare        124525.0        313128.0         188603.0       151.5
  Leitrim         89902.0        220569.0         130667.0       145.3
   Offaly        112410.0        274347.0         161937.0       144.1
    Laois        128642.0        309471.0         180829.0       140.6
Waterford        130365.0        313107.0         182742.0       140.2
  Wexford        133717.0        320233.0         186516.0       139.5
 Limerick        138148.0        329093.0         190945.0       138.2
   Carlow        128376.0        290245.0         161869.0       126.1
    Louth        144916.0        327667.0  

The fastest growing counties are not the most expensive ones - in fact 
the opposite is true:

- **Longford** recorded the highest growth at **187.1%** - prices 
nearly tripled from €72,440 to €207,969 over 10 years
- **Westmeath (174.4%)** and **Roscommon (162.9%)** follow, both 
Midlands counties that started from a very low base in 2015
- **Dublin recorded the lowest growth at 65.4%** despite having the 
highest absolute increase (€228,626)

This pattern reflects a classic catch-up effect - counties that were 
severely undervalued after the crash have grown faster in percentage 
terms as buyers priced out of Dublin and other expensive counties 
looked further afield. The Midlands and West counties (Longford, 
Westmeath, Roscommon, Cavan) all recorded growth above 160%, 
suggesting a significant rebalancing of the Irish property market 
over the past decade driven by affordability pressures in Dublin 
and its commuter belt.

### Query 10 - Affordability Analysis by County

This query calculates what percentage of transactions in each county 
fall below €300,000 - a commonly cited affordability threshold for 
first-time buyers in Ireland - giving a measure of how accessible 
each county's market is.

In [75]:
query10 = """
SELECT 
    County,
    COUNT(*) as total_transactions,
    SUM(CASE WHEN Price < 300000 THEN 1 ELSE 0 END) as under_300k,
    ROUND(SUM(CASE WHEN Price < 300000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_under_300k,
    ROUND(AVG(Price), 0) as avg_price
FROM transactions
WHERE Year = (SELECT MAX(Year) FROM transactions)
GROUP BY County
ORDER BY pct_under_300k DESC
"""

result10 = pd.read_sql(query10, conn)
print("Affordability Analysis by County (Latest Year):")
print(result10.to_string(index=False))

Affordability Analysis by County (Latest Year):
   County  total_transactions  under_300k  pct_under_300k  avg_price
 Longford                 153         128            83.7   204414.0
  Donegal                 513         386            75.2   238045.0
  Leitrim                 171         127            74.3   221470.0
Roscommon                 248         178            71.8   236181.0
     Mayo                 449         322            71.7   249527.0
 Monaghan                 155         109            70.3   247819.0
    Cavan                 276         182            65.9   264175.0
Tipperary                 532         327            61.5   275912.0
    Sligo                 312         188            60.3   290188.0
   Offaly                 263         151            57.4   280076.0
   Carlow                 169          94            55.6   300863.0
 Limerick                 683         348            51.0   310293.0
    Clare                 468         235            50

The affordability gap between Dublin and rural Ireland is stark:

- **Longford** is the most affordable county with 83.7% of transactions 
below €300,000 - over 4 in 5 properties are within this threshold
- **Donegal (75.2%), Leitrim (74.3%), Roscommon (71.8%)** and **Mayo (71.7%)** 
are the next most affordable counties, all in the West and North West
- **Dublin** is the least affordable county with only **8.4%** of 
transactions below €300,000 - meaning over 9 in 10 Dublin properties 
exceed this threshold
- **Wicklow (11.6%)** and **Kildare (13.3%)** are similarly unaffordable, 
reflecting their position in the Greater Dublin Area commuter belt

For a first-time buyer with a budget of €300,000, the data shows they 
have dramatically different options depending on location - from a wide 
choice of properties in Longford, Donegal and Leitrim, to an extremely 
limited selection in Dublin, Wicklow and Kildare. This affordability 
analysis quantifies the scale of Ireland's regional housing inequality 
in practical terms for buyers.

### Query 11 - County Price Ranking Using CTE

A Common Table Expression (CTE) allows us to write cleaner, more readable 
SQL by breaking a complex query into named steps. Here we use a CTE to 
first calculate the average price per county, then rank counties by price 
in a separate step.

In [76]:
query11 = """
WITH county_avg AS (
    SELECT
        County,
        COUNT(*) as total_transactions,
        ROUND(AVG(Price), 0) as avg_price
    FROM transactions
    GROUP BY County
),
county_ranked AS (
    SELECT
        County,
        total_transactions,
        avg_price,
        RANK() OVER (ORDER BY avg_price DESC) as price_rank
    FROM county_avg
)
SELECT *
FROM county_ranked
ORDER BY price_rank
"""

result11 = pd.read_sql(query11, conn)
print("County Price Ranking (CTE + RANK):")
print(result11.to_string(index=False))

County Price Ranking (CTE + RANK):
   County  total_transactions  avg_price  price_rank
   Dublin              235429   443132.0           1
  Wicklow               25382   381749.0           2
  Kildare               39786   320194.0           3
    Meath               32067   291689.0           4
     Cork               82350   269184.0           5
   Galway               36458   251591.0           6
 Kilkenny               11646   235888.0           7
    Louth               20884   230878.0           8
 Limerick               27934   212007.0           9
  Wexford               25843   210414.0          10
Waterford               20202   207757.0          11
    Laois               12277   205551.0          12
    Kerry               20151   204505.0          13
Westmeath               14229   200220.0          14
    Clare               17093   198825.0          15
   Carlow                8026   195348.0          16
   Offaly                9312   183818.0          17
Tipperary  

Using a CTE to first calculate county averages and then apply RANK() 
gives us a clean, readable county league table across all years:

- **Dublin ranks 1st** with an average price of €443,132 across all 
748,051 transactions
- **Wicklow (2nd, €381,749)** and **Kildare (3rd, €320,194)** complete 
the top 3, confirming the Greater Dublin Area dominance
- **Longford ranks last (26th)** at €128,803 - less than 30% of Dublin's 
average price

The CTE approach makes this query easy to extend - for example, adding 
a second CTE step to filter only counties above or below a price 
threshold, without rewriting the aggregation logic.

### Query 12 - Top 3 Most Expensive Counties Per Year

This query uses ROW_NUMBER() partitioned by Year to identify the three 
most expensive counties in each year. This is a common interview question 
pattern - "find the top N records within each group" - and demonstrates 
advanced window function usage.

In [77]:
query12 = """
WITH county_year_avg AS (
    SELECT
        Year,
        County,
        ROUND(AVG(Price), 0) as avg_price,
        COUNT(*) as transactions
    FROM transactions
    GROUP BY Year, County
),
ranked AS (
    SELECT
        Year,
        County,
        avg_price,
        transactions,
        ROW_NUMBER() OVER (PARTITION BY Year ORDER BY avg_price DESC) as rank_in_year
    FROM county_year_avg
)
SELECT *
FROM ranked
WHERE rank_in_year <= 3
ORDER BY Year, rank_in_year
"""

result12 = pd.read_sql(query12, conn)
print("Top 3 Most Expensive Counties Per Year:")
print(result12.to_string(index=False))

Top 3 Most Expensive Counties Per Year:
 Year  County  avg_price  transactions  rank_in_year
 2010  Dublin   331272.0          6636             1
 2010 Wicklow   298994.0           689             2
 2010 Kildare   244517.0          1007             3
 2011  Dublin   313264.0          5576             1
 2011 Wicklow   254844.0           541             2
 2011 Kildare   229027.0           769             3
 2012  Dublin   284779.0          8360             1
 2012 Wicklow   238435.0           720             2
 2012 Kildare   192904.0          1030             3
 2013  Dublin   304071.0          9931             1
 2013 Wicklow   250977.0           921             2
 2013 Kildare   203026.0          1284             3
 2014  Dublin   337255.0         13798             1
 2014 Wicklow   265523.0          1310             2
 2014 Kildare   226510.0          1830             3
 2015  Dublin   349430.0         14787             1
 2015 Wicklow   289686.0          1367             2
 2015 

One of the most striking findings from this query is the **consistency** 
of the top 3:

- **Dublin, Wicklow, and Kildare have occupied the top 3 positions every 
single year from 2010 to 2026** without exception
- Dublin has held 1st place in every year, with average prices growing 
from €331,272 in 2010 to €591,104 in 2026 - a 78% increase
- Wicklow has held 2nd place consistently, growing from €298,994 to 
€534,350 over the same period
- Kildare has held 3rd place consistently, growing from €244,517 to 
€440,878

This remarkable consistency demonstrates how entrenched the Greater 
Dublin Area's price premium is. Despite 16 years of market cycles 
including a financial crash, a pandemic, and rapid price growth, the 
regional hierarchy of the Irish property market has remained completely 
stable at the top.

The ROW_NUMBER() PARTITION BY Year pattern used here is one of the most 
common advanced SQL interview questions - finding the top N records 
within each group.

### Query 13 - RANK vs DENSE_RANK vs ROW_NUMBER

A common interview question is: what is the difference between RANK(), 
DENSE_RANK(), and ROW_NUMBER()?

- **ROW_NUMBER()** - always assigns unique sequential numbers, no ties
- **RANK()** - assigns the same rank to ties, then skips numbers (1,1,3)
- **DENSE_RANK()** - assigns the same rank to ties, but does not skip (1,1,2)

This query demonstrates all three on county average prices so the 
difference is visible in the output.

In [78]:
query13 = """
WITH county_avg AS (
    SELECT
        County,
        ROUND(AVG(Price), 0) as avg_price
    FROM transactions
    WHERE Year = (SELECT MAX(Year) FROM transactions)
    GROUP BY County
)
SELECT
    County,
    avg_price,
    ROW_NUMBER() OVER (ORDER BY avg_price DESC) as row_number,
    RANK()       OVER (ORDER BY avg_price DESC) as rank,
    DENSE_RANK() OVER (ORDER BY avg_price DESC) as dense_rank
FROM county_avg
ORDER BY avg_price DESC
"""

result13 = pd.read_sql(query13, conn)
print("RANK vs DENSE_RANK vs ROW_NUMBER:")
print(result13.to_string(index=False))

RANK vs DENSE_RANK vs ROW_NUMBER:
   County  avg_price  row_number  rank  dense_rank
   Dublin   591104.0           1     1           1
  Wicklow   534350.0           2     2           2
  Kildare   440878.0           3     3           3
    Meath   420534.0           4     4           4
     Cork   389516.0           5     5           5
   Galway   385928.0           6     6           6
 Kilkenny   355888.0           7     7           7
    Louth   346927.0           8     8           8
    Laois   343113.0           9     9           9
    Kerry   327947.0          10    10          10
Waterford   317401.0          11    11          11
  Wexford   317041.0          12    12          12
    Clare   312290.0          13    13          13
 Limerick   310293.0          14    14          14
Westmeath   308027.0          15    15          15
   Carlow   300863.0          16    16          16
    Sligo   290188.0          17    17          17
   Offaly   280076.0          18    18          

In this dataset all three ranking functions produce identical results 
because no two counties share exactly the same average price in 2026 
- there are no ties to distinguish them.

To understand the difference, consider a hypothetical example where 
two counties both have an average price of €300,000:

| County | Price | ROW_NUMBER | RANK | DENSE_RANK |
|---|---|---|---|---|
| Cork | €350,000 | 1 | 1 | 1 |
| Galway | €300,000 | 2 | 2 | 2 |
| Limerick | €300,000 | 3 | 2 | 2 |
| Clare | €280,000 | 4 | 4 | 3 |

- **ROW_NUMBER** always assigns unique numbers - ties get different ranks arbitrarily
- **RANK** assigns the same rank to ties but skips the next number (jumps from 2 to 4)
- **DENSE_RANK** assigns the same rank to ties but does not skip numbers (goes 1,2,2,3)

### Query 14 - 3-Year Rolling Average Price

A rolling average smooths out year-to-year noise and reveals the 
underlying price trend more clearly. This query uses a window frame 
clause (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) to calculate a 
3-year rolling average for each year.

This is particularly useful for identifying whether the market is in 
a genuine upward trend or just experiencing short-term volatility.

In [79]:
query14 = """
WITH yearly_prices AS (
    SELECT
        Year,
        AVG(Price) AS avg_price
    FROM transactions
    GROUP BY Year
)
SELECT
    Year,
    ROUND(avg_price, 0) AS avg_price,
    ROUND(
        AVG(avg_price) OVER (
            ORDER BY Year
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        0
    ) AS rolling_3yr_avg
FROM yearly_prices
ORDER BY Year
"""

result14 = pd.read_sql(query14, conn)
print("3-Year Rolling Average Price:")
print(result14.to_string(index=False))

3-Year Rolling Average Price:
 Year  avg_price  rolling_3yr_avg
 2010   245871.0         245871.0
 2011   219203.0         232537.0
 2012   197976.0         221017.0
 2013   199508.0         205562.0
 2014   207053.0         201512.0
 2015   216941.0         207834.0
 2016   238107.0         220700.0
 2017   265615.0         240221.0
 2018   280112.0         261278.0
 2019   285237.0         276988.0
 2020   291242.0         285530.0
 2021   317496.0         297992.0
 2022   348395.0         319044.0
 2023   362266.0         342719.0
 2024   391403.0         367354.0
 2025   415516.0         389728.0
 2026   422981.0         409967.0


The rolling 3-year average smooths out year-to-year fluctuations and 
reveals the underlying trend more clearly:

- **2010-2013:** The rolling average falls consistently, confirming a 
genuine sustained price decline rather than short-term noise
- **2014-2026:** The rolling average rises consistently every single year, 
confirming a genuine sustained upward trend with no interruption - 
not even COVID-19 in 2020 broke the rolling average trend
- **2026:** The rolling average of €409,967 is slightly below the actual 
average of €422,981, suggesting recent prices are accelerating above 
the 3-year trend

The `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` window frame is what 
makes this possible - it tells SQL to look at the current row and the 
two rows before it when calculating the average, sliding forward one 
row at a time. This is a powerful technique for time-series smoothing 
in any dataset with a time dimension.

## Database Schema

### Entity Relationship Diagram

```
transactions
------------
Date          TEXT    - Sale completion date
County        TEXT    - Irish county (26 values)
Year          INTEGER - Extracted from Date
Month         INTEGER - Extracted from Date
Price         REAL    - Sale price in euros
Property Type TEXT    - 'New Build' or 'Second-Hand'
Eircode       TEXT    - Irish postal code (largely missing)
```

This is a single-table schema since the Property Price Register is a 
flat file containing one record per transaction. In a production 
environment this could be normalised into:

- **dim_county** - county reference table with region and population data
- **dim_property_type** - property type lookup
- **fact_transactions** - core transaction table with foreign keys

### Why SQLite?

SQLite was chosen for this analysis because:

- It requires no server setup - the entire database is a single file
- It is built into Python via the `sqlite3` module - no installation needed
- It supports standard SQL including window functions (LAG, RANK, ROW_NUMBER)
- It is ideal for analytics prototyping on datasets up to a few million rows
- The database file (`housing.db`) can be shared alongside the notebook

For a production environment with multiple users or real-time updates, 
PostgreSQL or a cloud data warehouse such as BigQuery or Snowflake would 
be more appropriate.

## Conclusion

This SQL analysis of 748,051 Irish property transactions demonstrates 
a complete analytical SQL workflow:

- **Database creation** - loading cleaned data into SQLite
- **Aggregations** - GROUP BY, AVG, COUNT, MIN, MAX
- **Conditional aggregation** - CASE WHEN for Dublin vs national comparisons
- **Window functions** - LAG(), RANK(), DENSE_RANK(), ROW_NUMBER() and 
ROWS BETWEEN for rolling averages
- **CTEs** - Common Table Expressions used in Queries 3, 11, 12, 13 and 14 
for readable, modular queries
- **Subqueries** - filtering to latest year dynamically
- **Business interpretation** - every query followed by findings and 
business implications

### Key SQL Findings

| Business Question | Answer |
|---|---|
| Most expensive county | Dublin (€591,104 avg in 2026) |
| Most affordable county | Longford (€204,414 avg in 2026) |
| Strongest price growth 2015-2025 | Longford (+187.1%) |
| Most active market | Dublin (235,429 transactions) |
| Most affordable county for buyers | Longford (83.7% under €300k) |
| Peak transaction month | December (89,524 transactions) |
| Top 3 counties every year since 2010 | Dublin, Wicklow, Kildare |
| Steepest price fall | 2011 (-10.8% year on year) |
| Strongest price growth year | 2017 (+11.6% year on year) |
| Rolling 3yr avg price (2026) | €409,967 |

### SQL Concepts Demonstrated

| Concept | Query |
|---|---|
| GROUP BY + Aggregations | Queries 1, 2, 6, 7, 8 |
| Subqueries | Queries 2, 7, 10, 13 |
| CASE WHEN | Queries 4, 5, 7, 10 |
| LAG() Window Function | Query 3 |
| RANK() | Query 11 |
| DENSE_RANK() | Query 13 |
| ROW_NUMBER() | Query 12, 13 |
| PARTITION BY | Query 12 |
| ROWS BETWEEN (Rolling Average) | Query 14 |
| CTEs | Queries 3, 11, 12, 13, 14 |
| JOIN | Query 9 |

### Next Steps
These SQL query results have been exported as CSVs and will be used 
to build an interactive Power BI dashboard in the next phase of 
this project.

In [80]:
# Save all queries to a .sql file for GitHub portfolio
sql_queries = """
-- Irish Property Price Register - SQL Analysis
-- Author: Navya Zacharia
-- Dataset: propertypriceregister.ie

-- Query 1: National Average Price and Transaction Volume by Year
SELECT 
    Year,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price
FROM transactions
GROUP BY Year
ORDER BY Year;

-- Query 2: Average Price by County (Latest Year)
SELECT 
    County,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price,
    ROUND(MIN(Price), 0) as min_price,
    ROUND(MAX(Price), 0) as max_price
FROM transactions
WHERE Year = (SELECT MAX(Year) FROM transactions)
GROUP BY County
ORDER BY avg_price DESC;

-- Query 3: Year on Year Price Growth (CTE version)
WITH yearly_prices AS (
    SELECT
        Year,
        AVG(Price) as avg_price
    FROM transactions
    GROUP BY Year
)
SELECT
    Year,
    ROUND(avg_price, 0) as avg_price,
    ROUND(avg_price - LAG(avg_price) OVER (ORDER BY Year), 0) as yoy_change,
    ROUND(
        (avg_price - LAG(avg_price) OVER (ORDER BY Year))
        / LAG(avg_price) OVER (ORDER BY Year) * 100,
        1
    ) as yoy_pct_change
FROM yearly_prices
ORDER BY Year;

-- Query 4: Dublin vs National Average Price by Year
SELECT 
    Year,
    ROUND(AVG(CASE WHEN County = 'Dublin' THEN Price END), 0) as dublin_avg,
    ROUND(AVG(Price), 0) as national_avg,
    ROUND(AVG(CASE WHEN County = 'Dublin' THEN Price END) - AVG(Price), 0) as dublin_premium
FROM transactions
GROUP BY Year
ORDER BY Year;

-- Query 5: New Build vs Second-Hand Price by Year
SELECT 
    Year,
    ROUND(AVG(CASE WHEN [Property Type] = 'New Build' THEN Price END), 0) as new_build_avg,
    ROUND(AVG(CASE WHEN [Property Type] = 'Second-Hand' THEN Price END), 0) as second_hand_avg,
    ROUND(AVG(CASE WHEN [Property Type] = 'New Build' THEN Price END) - 
          AVG(CASE WHEN [Property Type] = 'Second-Hand' THEN Price END), 0) as new_build_premium
FROM transactions
GROUP BY Year
ORDER BY Year;

-- Query 6: Top 10 Counties by Transaction Volume
SELECT 
    County,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price,
    ROUND(MIN(Price), 0) as min_price,
    ROUND(MAX(Price), 0) as max_price
FROM transactions
GROUP BY County
ORDER BY total_transactions DESC
LIMIT 10;

-- Query 7: Price Band Distribution
SELECT 
    CASE 
        WHEN Price < 100000 THEN 'Under €100k'
        WHEN Price < 200000 THEN '€100k - €200k'
        WHEN Price < 300000 THEN '€200k - €300k'
        WHEN Price < 400000 THEN '€300k - €400k'
        WHEN Price < 500000 THEN '€400k - €500k'
        ELSE 'Over €500k'
    END as price_band,
    COUNT(*) as transaction_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 1) as pct_of_total
FROM transactions
GROUP BY price_band
ORDER BY MIN(Price);

-- Query 8: Monthly Seasonality Analysis
SELECT 
    Month,
    COUNT(*) as total_transactions,
    ROUND(AVG(Price), 0) as avg_price
FROM transactions
GROUP BY Month
ORDER BY Month;

-- Query 9: County Price Growth Ranking (2015 to 2025)
SELECT 
    a.County,
    ROUND(a.avg_price_2015, 0) as avg_price_2015,
    ROUND(b.avg_price_2025, 0) as avg_price_2025,
    ROUND(b.avg_price_2025 - a.avg_price_2015, 0) as absolute_growth,
    ROUND((b.avg_price_2025 - a.avg_price_2015) / a.avg_price_2015 * 100, 1) as pct_growth
FROM 
    (SELECT County, AVG(Price) as avg_price_2015 
     FROM transactions WHERE Year = 2015 
     GROUP BY County) a
JOIN 
    (SELECT County, AVG(Price) as avg_price_2025 
     FROM transactions WHERE Year = 2025 
     GROUP BY County) b
ON a.County = b.County
ORDER BY pct_growth DESC;

-- Query 10: Affordability Analysis by County
SELECT 
    County,
    COUNT(*) as total_transactions,
    SUM(CASE WHEN Price < 300000 THEN 1 ELSE 0 END) as under_300k,
    ROUND(SUM(CASE WHEN Price < 300000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_under_300k,
    ROUND(AVG(Price), 0) as avg_price
FROM transactions
WHERE Year = (SELECT MAX(Year) FROM transactions)
GROUP BY County
ORDER BY pct_under_300k DESC;

-- Query 11: County Price Ranking Using CTE and RANK()
WITH county_avg AS (
    SELECT
        County,
        COUNT(*) as total_transactions,
        ROUND(AVG(Price), 0) as avg_price
    FROM transactions
    GROUP BY County
),
county_ranked AS (
    SELECT
        County,
        total_transactions,
        avg_price,
        RANK() OVER (ORDER BY avg_price DESC) as price_rank
    FROM county_avg
)
SELECT *
FROM county_ranked
ORDER BY price_rank;

-- Query 12: Top 3 Most Expensive Counties Per Year Using ROW_NUMBER()
WITH county_year_avg AS (
    SELECT
        Year,
        County,
        ROUND(AVG(Price), 0) as avg_price,
        COUNT(*) as transactions
    FROM transactions
    GROUP BY Year, County
),
ranked AS (
    SELECT
        Year,
        County,
        avg_price,
        transactions,
        ROW_NUMBER() OVER (PARTITION BY Year ORDER BY avg_price DESC) as rank_in_year
    FROM county_year_avg
)
SELECT *
FROM ranked
WHERE rank_in_year <= 3
ORDER BY Year, rank_in_year;

-- Query 13: RANK vs DENSE_RANK vs ROW_NUMBER Demonstration
WITH county_avg AS (
    SELECT
        County,
        ROUND(AVG(Price), 0) as avg_price
    FROM transactions
    WHERE Year = (SELECT MAX(Year) FROM transactions)
    GROUP BY County
)
SELECT
    County,
    avg_price,
    ROW_NUMBER() OVER (ORDER BY avg_price DESC) as row_number,
    RANK()       OVER (ORDER BY avg_price DESC) as rank,
    DENSE_RANK() OVER (ORDER BY avg_price DESC) as dense_rank
FROM county_avg
ORDER BY avg_price DESC;

-- Query 14: 3-Year Rolling Average Price
WITH yearly_prices AS (
    SELECT
        Year,
        AVG(Price) AS avg_price
    FROM transactions
    GROUP BY Year
)
SELECT
    Year,
    ROUND(avg_price, 0) AS avg_price,
    ROUND(
        AVG(avg_price) OVER (
            ORDER BY Year
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        0
    ) AS rolling_3yr_avg
FROM yearly_prices
ORDER BY Year;
"""

with open('housing_queries.sql', 'w') as f:
    f.write(sql_queries)

print("All 14 SQL queries saved to housing_queries.sql")

All 14 SQL queries saved to housing_queries.sql


In [81]:
# Export all query results as CSVs for Power BI dashboard

result1.to_csv('sql_national_trends.csv', index=False)
result2.to_csv('sql_county_prices.csv', index=False)
result3.to_csv('sql_yoy_growth.csv', index=False)
result4.to_csv('sql_dublin_vs_national.csv', index=False)
result5.to_csv('sql_newbuild_vs_secondhand.csv', index=False)
result7.to_csv('sql_price_bands.csv', index=False)
result9.to_csv('sql_county_growth.csv', index=False)
result10.to_csv('sql_affordability.csv', index=False)
result11.to_csv('sql_county_ranking.csv', index=False)
result12.to_csv('sql_top3_per_year.csv', index=False)
result13.to_csv('sql_rank_demonstration.csv', index=False)
result14.to_csv('sql_rolling_avg.csv', index=False)
print("All query results exported as CSVs for Power BI")

All query results exported as CSVs for Power BI
